# Information Gain & Decision-Tree Splits

Wiki reference for [decision-tree information gain](https://ml-viz-ruby.vercel.app/wiki/decision-tree-information-gain).

**The idea in one sentence.** A decision tree grows by picking, at each node, the split that
most **reduces impurity** — measured by Gini or entropy — where **information gain** is the
parent's impurity minus the weighted impurity of the children; it is always $\ge 0$, and the
greedy best-split search is the core of tree learning.

We implement Gini, entropy, and information gain from scratch, **validate the impurity
identities and that gain is non-negative**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')

## From-scratch split scorer

In [ ]:
def gini(labels):
    _, c = np.unique(labels, return_counts=True)
    p = c / c.sum()
    return 1 - (p**2).sum()

def entropy(labels):
    _, c = np.unique(labels, return_counts=True)
    p = c / c.sum()
    return -(p * np.log2(p + 1e-300)).sum()

def weighted_impurity(y, mask, fn=gini):
    n = len(y)
    return mask.sum()/n * fn(y[mask]) + (~mask).sum()/n * fn(y[~mask])

def info_gain(y, mask, fn=gini):
    return fn(y) - weighted_impurity(y, mask, fn)

# Loan-approval running example
ages   = np.array([22,25,28,30,33,36,40,45,50,60], dtype=float)
labels = np.array([0, 0, 0, 0, 1, 1, 0, 1, 1, 1])

# Full threshold scan
thresholds = (np.sort(ages)[:-1] + np.sort(ages)[1:]) / 2
gains = [(t, info_gain(labels, ages <= t)) for t in thresholds]

print(f"{'Threshold':>12} {'Gini Gain':>12}")
for t, g in gains:
    marker = " ← best" if g == max(g2 for _,g2 in gains) else ""
    print(f"{t:>12.1f} {g:>12.4f}{marker}")

### Validate: the impurity measures and gain behave as they must

A **pure** node has zero impurity; a balanced 50/50 node has Gini $0.5$ and entropy $1$ bit; a
**perfect** split (children fully pure) has information gain equal to the parent's impurity. We
confirm all four.

In [ ]:
assert gini(np.array([1, 1, 1])) == 0, 'a pure node has zero Gini impurity'
assert np.isclose(gini(np.array([0, 0, 1, 1])), 0.5), 'a 50/50 node has Gini 0.5'
assert np.isclose(entropy(np.array([0, 0, 1, 1])), 1.0), 'a 50/50 node has entropy 1 bit'
y = np.array([0, 0, 1, 1]); perfect = np.array([True, True, False, False])
assert np.isclose(info_gain(y, perfect), gini(y)), 'a perfect split has gain = parent impurity'
print('✅ impurity/gain identities hold: pure=0, balanced=max, perfect split recovers parent impurity')

## Visualize gain vs threshold

In [ ]:
ts, gs = zip(*gains)
plt.figure(figsize=(8, 4))
plt.plot(ts, gs, 'o-', color='#6366f1')
plt.axvline(ts[np.argmax(gs)], color='#f59e0b', linestyle='--', label=f'Best: {ts[np.argmax(gs)]:.1f}')
plt.xlabel('Split threshold (Age ≤ t)')
plt.ylabel('Gini information gain')
plt.title('Information gain vs threshold')
plt.legend(); plt.tight_layout(); plt.show()

### Validate: the best split has positive, non-negative gain

Information gain is **never negative** (splitting can't increase expected impurity), and the
best threshold on the age feature yields a strictly positive gain — a useful split. We confirm.

In [ ]:
ts, gs = zip(*gains)
print(f'best split: age <= {ts[int(np.argmax(gs))]:.1f} with gain {max(gs):.4f}')
assert all(g >= -1e-12 for g in gs), 'information gain is non-negative for every threshold'
assert max(gs) > 0, 'the best split yields a strictly positive information gain'
print('\n✅ the tree greedily picks the maximum-gain split at each node')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **high-cardinality bias** | unique-ID features maximise gain but overfit (demo) — gain ratio |
| **greedy is myopic** | the locally-best split isn't globally optimal |
| **no stopping rule** | trees grow until pure and overfit — prune / limit depth |
| **Gini vs entropy** | similar in practice; entropy is costlier (log) |
| **unstable** | small data changes flip splits — ensembles (RF) stabilise |

Demo: a unique-ID feature beats every real split on gain alone.

In [ ]:
# The classic information-gain gotcha: it is BIASED toward high-cardinality features. A feature
# with a unique value per sample can split every point into its own pure leaf, giving the
# MAXIMUM possible gain (the full parent impurity) — yet it generalises to nothing (pure
# memorisation). We show a unique-ID 'feature' beats every real split on gain alone.
parent = gini(labels)
id_gain = parent - 0.0        # unique-ID split -> every leaf pure -> weighted impurity 0
best_real = max(g for _, g in gains)
print(f'best real-feature gain = {best_real:.3f};  unique-ID split gain = {id_gain:.3f}')
assert id_gain >= best_real, 'a unique-ID feature maximises gain yet generalises to nothing'
print('\nInformation gain favours high-cardinality features -> use GAIN RATIO (C4.5) or Gini with limits.')

## ✏️ Your turn

**Task:** Add a second binary feature `income` = [30,45,50,60,35,70,40,80,90,65] (in thousands). Find the best *single* split across *both* features.

**Extension:** Build a depth-2 decision tree by hand: find the best split at the root, then the best split for each child node.

In [ ]:
# TODO(you): add income feature and find best split across both features
# income = np.array([30,45,50,60,35,70,40,80,90,65], dtype=float)
# For each feature + threshold, compute info_gain and track the best

In [ ]:
# best_gain should be > 0.3333 (better than age alone? or not?)
# assert best_gain >= 0.3333

<details><summary>Solution</summary>

```python
income = np.array([30,45,50,60,35,70,40,80,90,65], dtype=float)
features = {'age': ages, 'income': income}
best = (0, None, None)
for name, feat in features.items():
    for t in (np.sort(feat)[:-1]+np.sort(feat)[1:])/2:
        g = info_gain(labels, feat <= t)
        if g > best[0]:
            best = (g, name, t)
print(f'Best: {best[1]} <= {best[2]:.1f}, gain={best[0]:.4f}')
# Age <= 31.5 still wins at 0.3333
```
</details>

## Key takeaways

- **Impurity → gain:** split to reduce Gini/entropy; gain = parent impurity − weighted child
  impurity (verified).
- **Gain is non-negative:** splitting never increases expected impurity; the best split is
  strictly positive here (verified).
- **Greedy best-split** is the core of tree learning — evaluate every feature/threshold.
- **High-cardinality bias:** raw gain favours ID-like features (demo) — use gain ratio.